# TRABAJO PRÁCTICO 1: PARTE 2 - IDENTIFICACIÓN DE SISTEMAS Y COHERENCIA CUADRÁTICA

**Integrantes:** Ferreyra Florencia, González Tomás, Molina Lara y Scafati Jerónimo  
**Fecha:** 03/07/2026  
**Cátedra:** Procesamiento Digital de Señales (DSP)  

---


## Introducción

La segunda parte del trabajo práctico consiste en la **identificación de sistemas lineales e invariantes en el tiempo (LTI)** desconocidos mediante analizadores de doble canal. A partir de señales de entrada $x[n]$ y salida $y[n]$, se estimará la respuesta en frecuencia $H(\omega)$ y la **coherencia cuadrática** $\gamma_{xy}^2(\omega)$ para diagnosticar la linealidad del sistema a lo largo del espectro frecuencial.


In [ ]:
import sys
from pathlib import Path

# Agregar directorio del proyecto para importar las funciones académicas
project_root = Path.cwd()
while not (project_root / "faculty").exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "faculty" / "final"))


In [ ]:
import matplotlib.pyplot as plt
from functions import (
    add_white_noise,
    apply_fir,
    compute_fft,
    evaluar_coherencia,
    generate_pure_tones,
    identificar_sistema,
    load_fir_coefficients,
    plot_coherence,
    plot_frequency_response,
)


## 1. Carga y Simulación de Señales de Entrada y Salida

Para caracterizar el sistema, se utiliza una señal de entrada de prueba de banda ancha $x[n]$ (suma de tonos puros en $500\text{ Hz}$, $1000\text{ Hz}$ y $5000\text{ Hz}$ mezclada con ruido blanco gaussiano). La señal de salida $y[n]$ es el resultado de procesar la entrada a través de un sistema LTI pasabajos de corte en $1000\text{ Hz}$ al que se le suma ruido de medición no correlacionado en el canal de salida.


In [ ]:
fs = 44100
duracion = 2.0

# 1. Generación de señal de entrada x[n] de banda ancha
x_limpia = generate_pure_tones(frequencies=[500.0, 1000.0, 2500.0, 5000.0], amplitudes=[1.0, 0.8, 0.6, 0.4], fs=fs, duration=duracion)
x = add_white_noise(x_limpia, snr_db=25.0)

# 2. Procesamiento por el sistema LTI (Filtro FIR Hamming 1000 Hz) + Ruido de medición en la salida
coefs_path = project_root / "archivos" / "fir_hamming_1000Hz.npy"
h_true = load_fir_coefficients(str(coefs_path))

y_filtrada = apply_fir(x, h_true)
y = add_white_noise(y_filtrada, snr_db=15.0)

# 3. Visualización Espectral de Entrada y Salida
freqs_x, mag_x = compute_fft(x, fs)
freqs_y, mag_y = compute_fft(y, fs)

plt.figure(figsize=(12, 5))
plt.plot(freqs_x, mag_x, label="Espectro Entrada X(w)", alpha=0.7)
plt.plot(freqs_y, mag_y, label="Espectro Salida Y(w)", alpha=0.7, color='orange')
plt.title("Caracterización Espectral: Señal de Entrada vs Salida")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud")
plt.xlim(0, 8000)
plt.legend()
plt.grid(True)
plt.show()


## 2. Identificación de la Respuesta en Frecuencia $H(\omega)$

La respuesta en frecuencia del sistema se estima utilizando el estimador $H_1(\omega)$, basado en la densidad espectral cruzada $G_{xy}(\omega)$ y la auto-densidad espectral $G_{xx}(\omega)$ promediadas mediante el método de Welch:

$$H_1(\omega) = \frac{G_{xy}(\omega)}{G_{xx}(\omega)}$$

El promediado espectral reduce la varianza del ruido no correlacionado presente en la medición.


In [ ]:
# Estimación de H(w) con ventana Welch de 1024 muestras
freqs_h, H_est = identificar_sistema(x, y, fs=fs, window_size=1024)

# Gráfico de Respuesta en Frecuencia Estimada (Módulo acotado a [-60, 5] dB y Fase desenrollada)
fig_h = plot_frequency_response(freqs_h, H_est, title="Respuesta en Frecuencia Estimada H1(w)", ylim=(-60, 5))
plt.show()


### Análisis de la Respuesta en Frecuencia Estimada:
- **Banda de paso ($< 1000\text{ Hz}$)**: El sistema exhibe una ganancia unitaria ($0\text{ dB}$) con fase estrictamente lineal, lo que confirma que las componentes frecuenciales en la banda de paso sufren un retardo de grupo constante sin distorsión de amplitud.
- **Banda de parada ($> 1000\text{ Hz}$)**: Presenta una fuerte atenuación ($> 40\text{ dB}$), propia de un filtro pasabajos de alto orden.


## 3. Coherencia Cuadrática $\gamma_{xy}^2(\omega)$

La coherencia cuadrática permite evaluar la grado de relación lineal entre la entrada y la salida para cada frecuencia:

$$\gamma_{xy}^2(\omega) = \frac{|G_{xy}(\omega)|^2}{G_{xx}(\omega)\, G_{yy}(\omega)}$$

Sus valores se acotan estrictamente en el intervalo $[0, 1]$.


In [ ]:
# Cómputo de Coherencia Cuadrática con ventana Welch de 1024 muestras
freqs_coh, coh_val = evaluar_coherencia(x, y, fs=fs, window_size=1024)

# Visualización gráfica de la coherencia con umbral de linealidad en 0.9
fig_coh = plot_coherence(freqs_coh, coh_val, title="Coherencia Cuadrática gamma_xy^2(w)")
plt.show()


## 4. Preguntas Obligatorias de la Cátedra

### 1. ¿Por qué la coherencia vale 1 cuando la relación es perfectamente lineal, dada la forma en que se calcula?

**Demostración matemática:**  
Si un sistema es estrictamente lineal e invariante en el tiempo (LTI) y no existe ruido aditivo externo, la salida en el dominio frecuencial viene dada por $Y(\omega) = H(\omega)X(\omega)$.  
Las densidades espectrales resultan:  
$$G_{xx}(\omega) = X^*(\omega)X(\omega) = |X(\omega)|^2$$
$$G_{xy}(\omega) = X^*(\omega)Y(\omega) = X^*(\omega)[H(\omega)X(\omega)] = H(\omega)|X(\omega)|^2 = H(\omega)G_{xx}(\omega)$$
$$G_{yy}(\omega) = Y^*(\omega)Y(\omega) = [H^*(\omega)X^*(\omega)][H(\omega)X(\omega)] = |H(\omega)|^2 G_{xx}(\omega)$$

Sustituyendo estas expresiones en la ecuación de la coherencia cuadrática:

$$\gamma_{xy}^2(\omega) = \frac{|G_{xy}(\omega)|^2}{G_{xx}(\omega)\, G_{yy}(\omega)} = \frac{|H(\omega)G_{xx}(\omega)|^2}{G_{xx}(\omega) \cdot [|H(\omega)|^2 G_{xx}(\omega)]} = \frac{|H(\omega)|^2 G_{xx}^2(\omega)}{|H(\omega)|^2 G_{xx}^2(\omega)} = 1$$

**Interpretación física:**  
Cuando $\gamma_{xy}^2(\omega) = 1$, la totalidad de la energía observada en la salida en esa frecuencia está completamente explicada por la entrada a través de una transformación lineal $H(\omega)$. Si existe ruido no correlacionado $n[n]$ o distorsión no lineal ($y = x^2$), $G_{yy}(\omega)$ incorpora potencia no relacionada con $x[n]$, aumentando el denominador y haciendo que $\gamma_{xy}^2(\omega) < 1$.

---

### 2. ¿Cómo es la linealidad del sistema en distintas partes del espectro?

- **En la Banda de Paso ($0$ a $1000\text{ Hz}$)**: La coherencia es extremadamente alta ($\gamma_{xy}^2(\omega) \approx 1.0$), lo que demuestra que en esa zona del espectro la relación entrada-salida es fuertemente lineal y la señal de salida $y[n]$ proviene casi en su totalidad de la entrada $x[n]$.
- **En la Banda de Parada ($> 1000\text{ Hz}$)**: La coherencia cae abruptamente hacia cero ($\gamma_{xy}^2(\omega) \to 0$). Debido a la atenuación del filtro pasabajos, la componente lineal transmitida cae por debajo del piso de ruido del canal de medición. Por ende, la señal presente a la salida en altas frecuencias es principalmente ruido blanco no correlacionado, destruyendo la relación lineal entre $x$ e $y$.

---

### 3. ¿Qué tipos de sistemas físicos reales podrían dar lugar a ese comportamiento?

1. **Sistemas de Audio y Electroacústicos (Altavoces y Micrófonos)**:  
   Tienen un ancho de banda limitado por sus propiedades mecánicas (masa del diafragma, inductancia de bobina). En su banda pasante responden de forma lineal (coherencia 1), mientras que fuera de su rango útil la señal transmitida cae y predomina el ruido térmico del preamplificador o el ruido ambiental.

2. **Canales de Transmisión Filtrados con Ruido Ambiental (AWGN)**:  
   Cables de comunicación o líneas de transmisión con filtros antialiasing donde la señal útil está confinada a una banda y el resto del espectro es dominado por ruido gaussiano blanco no correlacionado.

3. **Recintos Acústicos y Aislamiento Sonoro**:  
   Paredes o tabiques de aislación acustica atenúan fuertemente las frecuencias altas mientras transmiten las frecuencias bajas. En altas frecuencias, el sonido registrado al otro lado proviene de fuentes difusas o ruido propio de fondo en lugar del emisor directo.


## 5. Conclusión General

La caracterización no paramétrica mediante el estimador $H_1(\omega)$ y la **coherencia cuadrática** $\gamma_{xy}^2(\omega)$ constituye una metodología fundamental en el procesamiento digital de señales. Permite no solo identificar la magnitud y fase del sistema LTI, sino también verificar la validez de la medición e identificar las regiones del espectro donde el modelo lineal es confiable frente a perturbaciones o atenuación profunda.
